# ARC-AGI-2 Grand Prize Solver

**Target**: 85%+ on ARC-AGI-2 private eval → $700k Grand Prize

**Architecture**: Hybrid DSL program search + neural TTT fallback

**Strategy**:
1. **DSL Search** — Compositional search over grid transformation primitives, verified against training pairs
2. **Neural TTT** — Test-time trained encoder-decoder for tasks DSL can't solve
3. **Ensemble** — Take the best of both approaches per task

**Constraints**: Kaggle notebook, 12hr runtime, no internet, L4x4 GPU

In [ ]:
# ============================================================
# Cell 1: Environment Setup
# ============================================================
import subprocess, sys, os

# Detect environment
ON_KAGGLE = os.path.exists('/kaggle/input')
ON_COLAB = 'COLAB_GPU' in os.environ

if ON_KAGGLE:
    DATA_DIR = '/kaggle/input/arc-prize-2025'
elif ON_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'arckit'])
    DATA_DIR = None  # Will use arckit
else:
    DATA_DIR = None  # Local dev — use arckit

import json
import time
import copy
import itertools
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional, Callable, Any
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Environment: {"Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "Local"}')
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

In [ ]:
# ============================================================
# Cell 2: Data Loading
# ============================================================

@dataclass
class ARCPair:
    input: np.ndarray
    output: np.ndarray

@dataclass
class ARCTask:
    task_id: str
    train: List[ARCPair]
    test: List[ARCPair]  # output may be None for competition test set

def load_kaggle_tasks(data_dir: str) -> Tuple[Dict[str, ARCTask], Dict[str, ARCTask], Dict[str, ARCTask]]:
    """Load tasks from Kaggle competition directory."""
    def parse_challenges(chal_path, sol_path=None):
        with open(chal_path) as f:
            challenges = json.load(f)
        solutions = {}
        if sol_path and os.path.exists(sol_path):
            with open(sol_path) as f:
                solutions = json.load(f)
        tasks = {}
        for tid, task_data in challenges.items():
            train = [ARCPair(np.array(p['input']), np.array(p['output'])) for p in task_data['train']]
            test = []
            for i, p in enumerate(task_data['test']):
                inp = np.array(p['input'])
                out = np.array(solutions[tid][i]) if tid in solutions else None
                test.append(ARCPair(inp, out))
            tasks[tid] = ARCTask(tid, train, test)
        return tasks

    d = Path(data_dir)
    train = parse_challenges(d / 'arc-agi_training_challenges.json', d / 'arc-agi_training_solutions.json')
    val = parse_challenges(d / 'arc-agi_evaluation_challenges.json', d / 'arc-agi_evaluation_solutions.json')
    test = parse_challenges(d / 'arc-agi_test_challenges.json')
    return train, val, test

def load_arckit_tasks():
    """Load via arckit package (for local/Colab dev)."""
    import arckit
    train_set, eval_set = arckit.load_data()
    def convert(tasks):
        result = {}
        for t in tasks:
            train = [ARCPair(np.array(i), np.array(o)) for i, o in zip(t.train_inputs, t.train_outputs)]
            test = [ARCPair(np.array(i), np.array(o)) for i, o in zip(t.test_inputs, t.test_outputs)]
            result[t.id] = ARCTask(t.id, train, test)
        return result
    return convert(train_set), convert(eval_set), {}

# Load data
if DATA_DIR and os.path.exists(DATA_DIR):
    train_tasks, eval_tasks, test_tasks = load_kaggle_tasks(DATA_DIR)
else:
    train_tasks, eval_tasks, test_tasks = load_arckit_tasks()

print(f'Training tasks: {len(train_tasks)}')
print(f'Evaluation tasks: {len(eval_tasks)}')
print(f'Test tasks: {len(test_tasks)}')

---
## Part 1: DSL Program Search

The core insight: ARC gives you training examples to **verify** against. We can try thousands of candidate programs and keep only those that produce exact matches on ALL training pairs. This is brute-force but reliable — if a program passes all training pairs, it very likely generalizes to the test input.

In [ ]:
# ============================================================
# Cell 3: Grid Analysis Utilities
# ============================================================

def grid_colors(g: np.ndarray) -> set:
    return set(g.flatten().tolist())

def grid_bg(g: np.ndarray) -> int:
    """Most common color = background."""
    c = Counter(g.flatten().tolist())
    return c.most_common(1)[0][0]

def extract_objects(g: np.ndarray, bg: int = 0) -> List[dict]:
    """Flood-fill connected components."""
    h, w = g.shape
    visited = np.zeros_like(g, dtype=bool)
    objects = []
    for r in range(h):
        for c in range(w):
            if visited[r, c] or g[r, c] == bg:
                continue
            color = g[r, c]
            cells = []
            stack = [(r, c)]
            while stack:
                cr, cc = stack.pop()
                if 0 <= cr < h and 0 <= cc < w and not visited[cr, cc] and g[cr, cc] == color:
                    visited[cr, cc] = True
                    cells.append((cr, cc))
                    stack.extend([(cr-1,cc),(cr+1,cc),(cr,cc-1),(cr,cc+1)])
            rows = [r for r, c in cells]
            cols = [c for r, c in cells]
            objects.append({
                'color': int(color), 'cells': cells,
                'bbox': (min(rows), min(cols), max(rows)+1, max(cols)+1),
                'size': len(cells),
            })
    return objects

def crop_to_content(g: np.ndarray, bg: int = 0) -> np.ndarray:
    """Crop to bounding box of non-background cells."""
    rows, cols = np.where(g != bg)
    if len(rows) == 0:
        return g.copy()
    return g[rows.min():rows.max()+1, cols.min():cols.max()+1].copy()

def get_object_subgrid(g: np.ndarray, obj: dict) -> np.ndarray:
    """Extract the bounding box subgrid for an object."""
    r0, c0, r1, c1 = obj['bbox']
    return g[r0:r1, c0:c1].copy()

def analyze_task(task: ARCTask) -> dict:
    """Compute structural features of a task to guide search."""
    info = {'same_shape': True, 'output_smaller': True, 'output_larger': True,
            'same_colors': True, 'fixed_output_size': True}
    out_shapes = set()
    for p in task.train:
        if p.input.shape != p.output.shape:
            info['same_shape'] = False
        if p.output.shape[0] > p.input.shape[0] or p.output.shape[1] > p.input.shape[1]:
            info['output_smaller'] = False
        if p.output.shape[0] < p.input.shape[0] or p.output.shape[1] < p.input.shape[1]:
            info['output_larger'] = False
        if grid_colors(p.output) - grid_colors(p.input):
            info['same_colors'] = False
        out_shapes.add(p.output.shape)
    info['fixed_output_size'] = len(out_shapes) == 1
    info['output_shape'] = out_shapes.pop() if len(out_shapes) == 1 else None
    return info

print('Grid utilities loaded.')

In [ ]:
# ============================================================
# Cell 4: DSL Primitives — Atomic Grid Transformations
# ============================================================
# Each primitive: np.ndarray → np.ndarray (or None if not applicable)

def identity(g): return g.copy()

# --- Geometric transforms ---
def rot90(g): return np.rot90(g, 1).copy()
def rot180(g): return np.rot90(g, 2).copy()
def rot270(g): return np.rot90(g, 3).copy()
def flip_h(g): return np.fliplr(g).copy()
def flip_v(g): return np.flipud(g).copy()
def transpose(g): return g.T.copy()
def transpose_anti(g): return np.rot90(g.T, 2).copy()

# --- Cropping ---
def crop_bg(g):
    bg = grid_bg(g)
    return crop_to_content(g, bg)

def crop_bg0(g):
    return crop_to_content(g, 0)

# --- Tiling ---
def tile_2x2(g): return np.tile(g, (2, 2))
def tile_3x3(g): return np.tile(g, (3, 3))
def tile_2x1(g): return np.tile(g, (2, 1))
def tile_1x2(g): return np.tile(g, (1, 2))

# --- Scaling ---
def upscale_2x(g): return np.repeat(np.repeat(g, 2, axis=0), 2, axis=1)
def upscale_3x(g): return np.repeat(np.repeat(g, 3, axis=0), 3, axis=1)
def downscale_2x(g):
    h, w = g.shape
    if h % 2 or w % 2: return None
    return g[::2, ::2].copy()
def downscale_3x(g):
    h, w = g.shape
    if h % 3 or w % 3: return None
    return g[::3, ::3].copy()

# --- Color transforms ---
def swap_bg_most(g):
    """Swap background (most common) with second most common color."""
    c = Counter(g.flatten().tolist())
    mc = c.most_common(2)
    if len(mc) < 2: return g.copy()
    bg, fg = mc[0][0], mc[1][0]
    r = g.copy()
    r[g == bg] = fg
    r[g == fg] = bg
    return r

def invert_colors(g):
    """Map color c → 9-c for non-zero cells."""
    r = g.copy()
    mask = r > 0
    r[mask] = 9 - r[mask]
    return r

def replace_with_most(g):
    """Replace all non-background with the single most common non-bg color."""
    bg = grid_bg(g)
    non_bg = g[g != bg]
    if len(non_bg) == 0: return g.copy()
    mc = Counter(non_bg.tolist()).most_common(1)[0][0]
    r = g.copy()
    r[r != bg] = mc
    return r

# --- Symmetry / mirroring ---
def mirror_h(g):
    """Concatenate grid with its horizontal mirror."""
    return np.concatenate([g, np.fliplr(g)], axis=1)

def mirror_v(g):
    return np.concatenate([g, np.flipud(g)], axis=0)

def mirror_both(g):
    top = np.concatenate([g, np.fliplr(g)], axis=1)
    return np.concatenate([top, np.flipud(top)], axis=0)

# --- Fill operations ---
def fill_border(g):
    """Fill a 1-cell border with the most common non-bg color."""
    bg = grid_bg(g)
    non_bg = g[g != bg]
    if len(non_bg) == 0: return g.copy()
    fc = Counter(non_bg.tolist()).most_common(1)[0][0]
    r = g.copy()
    r[0, :] = fc; r[-1, :] = fc; r[:, 0] = fc; r[:, -1] = fc
    return r

def hollow_interior(g):
    """Set interior non-border cells to background."""
    bg = grid_bg(g)
    r = np.full_like(g, bg)
    r[0, :] = g[0, :]; r[-1, :] = g[-1, :]
    r[:, 0] = g[:, 0]; r[:, -1] = g[:, -1]
    return r

# --- Gravity ---
def gravity_down(g):
    """Drop all non-bg cells to the bottom of each column."""
    bg = grid_bg(g)
    r = np.full_like(g, bg)
    for c in range(g.shape[1]):
        col = g[:, c]
        non_bg = col[col != bg]
        if len(non_bg) > 0:
            r[-len(non_bg):, c] = non_bg
    return r

def gravity_left(g):
    bg = grid_bg(g)
    r = np.full_like(g, bg)
    for row in range(g.shape[0]):
        vals = g[row][g[row] != bg]
        if len(vals) > 0:
            r[row, :len(vals)] = vals
    return r

# --- Extract largest object ---
def extract_largest(g):
    """Return the bounding box of the largest connected object."""
    bg = grid_bg(g)
    objs = extract_objects(g, bg)
    if not objs: return g.copy()
    largest = max(objs, key=lambda o: o['size'])
    return get_object_subgrid(g, largest)

def extract_smallest(g):
    bg = grid_bg(g)
    objs = extract_objects(g, bg)
    if not objs: return g.copy()
    smallest = min(objs, key=lambda o: o['size'])
    return get_object_subgrid(g, smallest)

# --- Mask / boolean ops ---
def to_binary(g):
    """Non-bg → 1, bg → 0."""
    bg = grid_bg(g)
    return (g != bg).astype(int)

def count_colors_per_cell(g):
    """Replace each cell with the count of that color in the grid."""
    c = Counter(g.flatten().tolist())
    r = np.zeros_like(g)
    for color, cnt in c.items():
        r[g == color] = cnt % 10  # Keep in 0-9 range
    return r

# --- Sort rows/cols ---
def sort_rows(g):
    """Sort each row independently."""
    return np.sort(g, axis=1)

def sort_cols(g):
    return np.sort(g, axis=0)

# Build the primitive catalog
PRIMITIVES = {
    'identity': identity,
    'rot90': rot90, 'rot180': rot180, 'rot270': rot270,
    'flip_h': flip_h, 'flip_v': flip_v,
    'transpose': transpose, 'transpose_anti': transpose_anti,
    'crop_bg': crop_bg, 'crop_bg0': crop_bg0,
    'tile_2x2': tile_2x2, 'tile_3x3': tile_3x3,
    'tile_2x1': tile_2x1, 'tile_1x2': tile_1x2,
    'upscale_2x': upscale_2x, 'upscale_3x': upscale_3x,
    'downscale_2x': downscale_2x, 'downscale_3x': downscale_3x,
    'swap_bg_most': swap_bg_most, 'invert_colors': invert_colors,
    'replace_with_most': replace_with_most,
    'mirror_h': mirror_h, 'mirror_v': mirror_v, 'mirror_both': mirror_both,
    'fill_border': fill_border, 'hollow_interior': hollow_interior,
    'gravity_down': gravity_down, 'gravity_left': gravity_left,
    'extract_largest': extract_largest, 'extract_smallest': extract_smallest,
    'to_binary': to_binary,
    'sort_rows': sort_rows, 'sort_cols': sort_cols,
}

print(f'DSL primitives: {len(PRIMITIVES)}')

In [ ]:
# ============================================================
# Cell 5: Parameterized Primitives (color-specific, object-specific)
# ============================================================

def make_color_replace_fns(task: ARCTask) -> Dict[str, Callable]:
    """Generate color-mapping primitives specific to this task's palette."""
    fns = {}
    # Gather all colors seen across all training pairs
    all_colors = set()
    for p in task.train:
        all_colors |= grid_colors(p.input)
        all_colors |= grid_colors(p.output)
    all_colors = sorted(all_colors)

    # Try to learn a consistent color mapping from train pairs
    # For each (src, dst) color pair, check if replacing src→dst in input matches output
    for src in all_colors:
        for dst in all_colors:
            if src == dst:
                continue
            def make_fn(s, d):
                def fn(g):
                    r = g.copy()
                    r[g == s] = d
                    return r
                return fn
            fns[f'recolor_{src}_to_{dst}'] = make_fn(src, dst)

    return fns

def make_input_output_ops(task: ARCTask) -> Dict[str, Callable]:
    """Generate operations that use patterns from the input-output relationship."""
    fns = {}

    # Check if output is a subgrid at a fixed offset
    # Check if output = input with specific rows/cols removed
    # Check if output dimensions are a simple function of input dimensions
    pair = task.train[0]
    ih, iw = pair.input.shape
    oh, ow = pair.output.shape

    # Top-left crop to output size
    if oh <= ih and ow <= iw:
        def crop_tl(g, h=oh, w=ow):
            return g[:h, :w].copy()
        fns[f'crop_to_{oh}x{ow}'] = crop_tl

    # Bottom-right crop
    if oh <= ih and ow <= iw:
        def crop_br(g, h=oh, w=ow):
            return g[-h:, -w:].copy()
        fns[f'crop_br_{oh}x{ow}'] = crop_br

    # Fixed output: if all outputs are identical, just return that
    outputs = [p.output for p in task.train]
    if all(np.array_equal(outputs[0], o) for o in outputs):
        fixed = outputs[0].copy()
        fns['fixed_output'] = lambda g, f=fixed: f.copy()

    return fns

print('Parameterized primitives loaded.')

In [ ]:
# ============================================================
# Cell 6: Program Search Engine
# ============================================================

def apply_program(g: np.ndarray, program: List[Callable]) -> Optional[np.ndarray]:
    """Apply a sequence of primitives to a grid. Returns None on failure."""
    result = g
    for fn in program:
        try:
            result = fn(result)
            if result is None:
                return None
            if not isinstance(result, np.ndarray):
                return None
            if result.size == 0 or result.ndim != 2:
                return None
            if max(result.shape) > 30:
                return None  # ARC grids max 30x30
        except Exception:
            return None
    return result

def verify_program(task: ARCTask, program: List[Callable]) -> bool:
    """Check if program produces exact match on ALL training pairs."""
    for pair in task.train:
        result = apply_program(pair.input, program)
        if result is None or not np.array_equal(result, pair.output):
            return False
    return True

def search_programs(
    task: ARCTask,
    max_depth: int = 3,
    time_limit: float = 30.0,
) -> Optional[Tuple[List[str], List[Callable]]]:
    """Search for a program that solves the task.

    Tries programs of increasing depth (1, 2, 3 primitives).
    Uses task analysis to prune the search space.
    Returns (program_names, program_fns) or None.
    """
    start = time.time()

    # Build task-specific primitives
    all_prims = dict(PRIMITIVES)
    all_prims.update(make_color_replace_fns(task))
    all_prims.update(make_input_output_ops(task))

    prim_names = list(all_prims.keys())
    prim_fns = [all_prims[n] for n in prim_names]
    n = len(prim_names)

    # Quick check: does the first training pair's output shape match analysis?
    info = analyze_task(task)

    # Depth 1: single primitives
    for i in range(n):
        if time.time() - start > time_limit:
            return None
        prog = [prim_fns[i]]
        if verify_program(task, prog):
            return ([prim_names[i]], prog)

    if max_depth < 2:
        return None

    # Depth 2: pairs of primitives
    # Pre-filter: only try a second primitive if the first changes the grid
    for i in range(n):
        if time.time() - start > time_limit:
            return None
        # Quick check: does first primitive produce valid intermediate?
        test_result = apply_program(task.train[0].input, [prim_fns[i]])
        if test_result is None:
            continue
        for j in range(n):
            if time.time() - start > time_limit:
                return None
            prog = [prim_fns[i], prim_fns[j]]
            if verify_program(task, prog):
                return ([prim_names[i], prim_names[j]], prog)

    if max_depth < 3:
        return None

    # Depth 3: triples (expensive — only try base primitives, not color-specific)
    base_names = list(PRIMITIVES.keys())
    base_fns = [PRIMITIVES[n] for n in base_names]
    nb = len(base_names)
    for i in range(nb):
        if time.time() - start > time_limit:
            return None
        r1 = apply_program(task.train[0].input, [base_fns[i]])
        if r1 is None:
            continue
        for j in range(nb):
            r2 = apply_program(r1, [base_fns[j]])
            if r2 is None:
                continue
            if time.time() - start > time_limit:
                return None
            for k in range(nb):
                if time.time() - start > time_limit:
                    return None
                prog = [base_fns[i], base_fns[j], base_fns[k]]
                if verify_program(task, prog):
                    return ([base_names[i], base_names[j], base_names[k]], prog)

    return None

print('Search engine loaded.')

In [ ]:
# ============================================================
# Cell 7: Evaluate DSL Search on Training Set (Sanity Check)
# ============================================================

def evaluate_dsl(tasks: Dict[str, ARCTask], time_per_task: float = 30.0, max_tasks: int = 100) -> dict:
    """Run DSL search on tasks and report solve rate."""
    solved = []
    failed = []
    total_time = 0
    task_list = list(tasks.values())[:max_tasks]

    for i, task in enumerate(task_list):
        t0 = time.time()
        result = search_programs(task, max_depth=2, time_limit=time_per_task)
        elapsed = time.time() - t0
        total_time += elapsed

        if result is not None:
            names, fns = result
            solved.append((task.task_id, names))
        else:
            failed.append(task.task_id)

        if (i + 1) % 25 == 0:
            print(f'  [{i+1}/{len(task_list)}] Solved: {len(solved)}, '
                  f'Rate: {len(solved)/(i+1):.1%}, Time: {total_time:.1f}s')

    rate = len(solved) / len(task_list) if task_list else 0
    print(f'\nDSL Results: {len(solved)}/{len(task_list)} = {rate:.1%}')
    print(f'Total time: {total_time:.1f}s ({total_time/len(task_list):.1f}s/task avg)')
    if solved:
        print(f'Example solutions:')
        for tid, names in solved[:5]:
            print(f'  {tid}: {" → ".join(names)}')
    return {'solved': solved, 'failed': failed, 'rate': rate}

print('--- DSL sanity check on training tasks ---')
dsl_results = evaluate_dsl(train_tasks, time_per_task=15.0, max_tasks=100)

---
## Part 2: Neural TTT Solver

For tasks the DSL can't solve, we fall back to the neural architecture:
- CNN encoder → Poincaré ball rule embedding → FiLM-conditioned decoder
- Pre-trained across all training tasks (meta-learning)
- Test-time trained on each specific task's training pairs

In [ ]:
# ============================================================
# Cell 8: Neural Model (self-contained)
# ============================================================

NUM_COLORS = 10
MAX_GRID = 30

def grid_to_tensor(g, device='cpu'):
    arr = np.array(g, dtype=np.int64)
    h, w = arr.shape
    oh = np.zeros((NUM_COLORS, h, w), dtype=np.float32)
    for c in range(NUM_COLORS):
        oh[c] = (arr == c).astype(np.float32)
    return torch.from_numpy(oh).to(device)

def pad_grid_t(t, size=MAX_GRID):
    c, h, w = t.shape
    if h >= size and w >= size:
        return t[:, :size, :size]
    p = torch.zeros(c, size, size, dtype=t.dtype, device=t.device)
    p[:, :h, :w] = t
    return p

def make_mask(h, w, size=MAX_GRID, device='cpu'):
    m = torch.zeros(size, size, device=device)
    m[:h, :w] = 1.0
    return m


class GridEncoder(nn.Module):
    def __init__(self, z_dim=128, hidden=256):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(NUM_COLORS, 64, 3, padding=1), nn.GroupNorm(8, 64), nn.GELU(),
            nn.Conv2d(64, 128, 3, padding=1), nn.GroupNorm(8, 128), nn.GELU(),
            nn.Conv2d(128, hidden, 3, padding=1), nn.GroupNorm(8, hidden), nn.GELU(),
        )
        self.attn_proj = nn.Linear(hidden, 1)
        self.proj = nn.Sequential(nn.Linear(hidden, z_dim), nn.LayerNorm(z_dim))

    def forward(self, grid, mask):
        feat = self.conv(grid)
        b, c, h, w = feat.shape
        ff = feat.view(b, c, h*w).permute(0, 2, 1)
        scores = self.attn_proj(ff).squeeze(-1)
        mf = mask.view(b, h*w)
        scores = scores.masked_fill(mf == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1).unsqueeze(-1)
        pooled = (ff * weights).sum(dim=1)
        return self.proj(pooled)


class PairEncoder(nn.Module):
    def __init__(self, z_dim=128):
        super().__init__()
        self.grid_enc = GridEncoder(z_dim=z_dim)
        self.pair_proj = nn.Sequential(
            nn.Linear(z_dim * 3, z_dim), nn.LayerNorm(z_dim), nn.GELU(),
            nn.Linear(z_dim, z_dim),
        )

    def forward(self, in_g, in_m, out_g, out_m):
        z_in = self.grid_enc(in_g, in_m)
        z_out = self.grid_enc(out_g, out_m)
        return self.pair_proj(torch.cat([z_in, z_out, z_out - z_in], dim=-1))


class PoincareBall:
    EPS = 1e-5
    @staticmethod
    def project(x, c=1.0, max_norm=0.95):
        norm = x.norm(dim=-1, keepdim=True)
        max_r = max_norm / (c ** 0.5)
        return torch.where(norm > max_r, x * max_r / norm, x)


class HyperbolicRuleEncoder(nn.Module):
    def __init__(self, z_dim=128, hyp_dim=32, curvature=1.0):
        super().__init__()
        self.curvature = curvature
        self.proj = nn.Sequential(nn.Linear(z_dim, z_dim), nn.GELU(), nn.Linear(z_dim, hyp_dim))

    def forward(self, z):
        return PoincareBall.project(self.proj(z), self.curvature)


class RuleConditioner(nn.Module):
    def __init__(self, z_dim, n_ch):
        super().__init__()
        self.gamma = nn.Linear(z_dim, n_ch)
        self.beta = nn.Linear(z_dim, n_ch)

    def forward(self, feat, z_rule):
        g = self.gamma(z_rule).unsqueeze(-1).unsqueeze(-1)
        b = self.beta(z_rule).unsqueeze(-1).unsqueeze(-1)
        return g * feat + b


class GridDecoder(nn.Module):
    def __init__(self, z_dim=128, hidden=256):
        super().__init__()
        self.z_dim = z_dim
        self.input_conv = nn.Sequential(
            nn.Conv2d(NUM_COLORS, 64, 3, padding=1), nn.GroupNorm(8, 64), nn.GELU(),
        )
        self.combine = nn.Conv2d(64 + z_dim, hidden, 1)
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Conv2d(hidden, hidden, 3, padding=1), nn.GroupNorm(8, hidden), nn.GELU())
            for _ in range(4)
        ])
        self.conditioners = nn.ModuleList([RuleConditioner(z_dim, hidden) for _ in range(4)])
        self.output = nn.Conv2d(hidden, NUM_COLORS, 1)

    def forward(self, z_rule, test_grid, test_mask):
        b = z_rule.shape[0]
        in_feat = self.input_conv(test_grid)
        z_sp = z_rule.unsqueeze(-1).unsqueeze(-1).expand(b, self.z_dim, MAX_GRID, MAX_GRID)
        h = self.combine(torch.cat([in_feat, z_sp], dim=1))
        for layer, cond in zip(self.layers, self.conditioners):
            h = layer(h)
            h = cond(h, z_rule)
        logits = self.output(h)
        return logits * test_mask.unsqueeze(1)

print('Neural model classes defined.')

In [ ]:
# ============================================================
# Cell 9: Neural Solver (Pre-training + TTT)
# ============================================================

class NeuralARCSolver(nn.Module):
    def __init__(self, z_dim=128, hyp_dim=32, hidden=256):
        super().__init__()
        self.pair_encoder = PairEncoder(z_dim=z_dim)
        self.grid_encoder = GridEncoder(z_dim=z_dim, hidden=hidden)
        self.rule_encoder = HyperbolicRuleEncoder(z_dim=z_dim, hyp_dim=hyp_dim)
        self.decoder = GridDecoder(z_dim=z_dim, hidden=hidden)
        self.rule_attn = nn.Sequential(nn.Linear(hyp_dim, 1), nn.Softmax(dim=0))
        self.rule_to_z = nn.Sequential(
            nn.Linear(hyp_dim, z_dim), nn.LayerNorm(z_dim), nn.GELU(), nn.Linear(z_dim, z_dim),
        )
        self.z_dim = z_dim

    def _enc(self, grid_np, device):
        t = grid_to_tensor(grid_np.tolist(), device)
        p = pad_grid_t(t)
        h, w = grid_np.shape
        m = make_mask(h, w, device=device)
        return p, m

    def infer_rule(self, pairs, device):
        z_pairs = []
        for ig, og in pairs:
            ip, im = self._enc(ig, device)
            op, om = self._enc(og, device)
            z = self.pair_encoder(ip.unsqueeze(0), im.unsqueeze(0), op.unsqueeze(0), om.unsqueeze(0))
            z_pairs.append(z.squeeze(0))
        z_stack = torch.stack(z_pairs)
        h_rules = self.rule_encoder(z_stack)
        weights = self.rule_attn(h_rules)
        h_agg = (weights * h_rules).sum(dim=0, keepdim=True)
        h_agg = PoincareBall.project(h_agg)
        return self.rule_to_z(h_agg)

    def predict(self, z_rule, test_input, device):
        ip, im = self._enc(test_input, device)
        logits = self.decoder(z_rule, ip.unsqueeze(0), im.unsqueeze(0))
        pred = logits.argmax(dim=1).squeeze(0)
        h, w = test_input.shape
        return pred[:h, :w].cpu().numpy().astype(int)

    def train_step(self, task_pairs, device):
        """One meta-learning step: leave-one-out prediction on training pairs."""
        total_loss = torch.tensor(0.0, device=device)
        for idx in range(len(task_pairs)):
            # Use all OTHER pairs to infer rule, predict THIS pair
            context = [p for i, p in enumerate(task_pairs) if i != idx]
            if not context:
                context = task_pairs
            target_in, target_out = task_pairs[idx]

            z_rule = self.infer_rule(context, device)
            ip, im = self._enc(target_in, device)
            op, om = self._enc(target_out, device)

            logits = self.decoder(z_rule, ip.unsqueeze(0), im.unsqueeze(0))
            target = op.argmax(dim=0).unsqueeze(0).long()
            loss = F.cross_entropy(logits, target, reduction='none')
            loss = (loss * om.unsqueeze(0)).sum() / om.sum().clamp(min=1)
            total_loss = total_loss + loss
        return total_loss / max(len(task_pairs), 1)

    def refine_on_task(self, pairs, device, steps=50, lr=1e-4):
        """Test-time training on a specific task."""
        params = list(self.decoder.parameters()) + list(self.rule_attn.parameters()) + list(self.rule_to_z.parameters())
        optimizer = optim.Adam(params, lr=lr)
        self.train()
        for _ in range(steps):
            optimizer.zero_grad()
            loss = self.train_step(pairs, device)
            loss.backward()
            optimizer.step()
        self.eval()
        return self.infer_rule(pairs, device)

    def solve_task(self, train_pairs, test_inputs, device, refine=True, refine_steps=50):
        """Solve a complete task. Returns list of [cand1, cand2] per test input."""
        self.to(device)

        # Save state for restoration after TTT
        state = copy.deepcopy(self.state_dict())

        if refine:
            z_rule = self.refine_on_task(train_pairs, device, steps=refine_steps)
        else:
            self.eval()
            with torch.no_grad():
                z_rule = self.infer_rule(train_pairs, device)

        results = []
        self.eval()
        with torch.no_grad():
            for ti in test_inputs:
                c1 = self.predict(z_rule, ti, device)
                results.append([c1, c1.copy()])  # duplicate as 2nd candidate

        # Restore pre-TTT state
        self.load_state_dict(state)
        return results

print('NeuralARCSolver defined.')

In [ ]:
# ============================================================
# Cell 10: Pre-train Neural Model
# ============================================================

def pretrain(model, tasks, device, epochs=30, lr=3e-4, max_tasks_per_epoch=200):
    """Meta-learning pre-training across many tasks."""
    model.to(device)
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    task_list = list(tasks.values())

    for epoch in range(epochs):
        np.random.shuffle(task_list)
        epoch_loss = 0
        n_tasks = min(len(task_list), max_tasks_per_epoch)

        for i in range(n_tasks):
            task = task_list[i]
            pairs = [(p.input, p.output) for p in task.train]
            if len(pairs) < 2:
                continue

            optimizer.zero_grad()
            try:
                loss = model.train_step(pairs, device)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                epoch_loss += loss.item()
            except Exception as e:
                continue  # Skip problematic tasks

        scheduler.step()
        avg = epoch_loss / max(n_tasks, 1)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'Epoch {epoch+1}/{epochs} — Loss: {avg:.4f} — LR: {scheduler.get_last_lr()[0]:.6f}')

    model.eval()
    return model

# Pre-train
print('Pre-training neural model...')
neural_model = NeuralARCSolver(z_dim=128, hyp_dim=32, hidden=256)
neural_model = pretrain(neural_model, train_tasks, DEVICE, epochs=30)
print('Pre-training complete.')

---
## Part 3: Hybrid Solver — DSL + Neural Ensemble

In [ ]:
# ============================================================
# Cell 11: Hybrid Solver
# ============================================================

def solve_task_hybrid(
    task: ARCTask,
    neural_model: NeuralARCSolver,
    device: str,
    dsl_time: float = 30.0,
    ttt_steps: int = 50,
) -> List[List[np.ndarray]]:
    """Solve a single task using DSL search + neural TTT fallback.

    Strategy:
    1. Try DSL search first (fast, exact when it works)
    2. If DSL fails, use neural TTT
    3. Return 2 candidates per test input
    """
    # --- Attempt 1: DSL search ---
    dsl_result = search_programs(task, max_depth=2, time_limit=dsl_time)

    if dsl_result is not None:
        names, fns = dsl_result
        results = []
        for tp in task.test:
            pred = apply_program(tp.input, fns)
            if pred is not None:
                results.append([pred, pred.copy()])
            else:
                # DSL found a program but it fails on test input — fall through to neural
                break
        else:
            return results  # DSL solved it

    # --- Attempt 2: Neural TTT ---
    train_pairs = [(p.input, p.output) for p in task.train]
    test_inputs = [p.input for p in task.test]

    try:
        neural_results = neural_model.solve_task(
            train_pairs, test_inputs, device,
            refine=True, refine_steps=ttt_steps,
        )
        return neural_results
    except Exception as e:
        # Last resort: return the input grid itself (better than nothing)
        return [[tp.input.copy(), tp.input.copy()] for tp in task.test]

print('Hybrid solver ready.')

In [ ]:
# ============================================================
# Cell 12: Evaluate on Eval Set
# ============================================================

def evaluate_hybrid(tasks: Dict[str, ARCTask], model, device,
                    max_tasks=None, dsl_time=20.0, ttt_steps=50):
    """Full evaluation with scoring."""
    task_list = list(tasks.values())
    if max_tasks:
        task_list = task_list[:max_tasks]

    correct = 0
    total = 0
    dsl_wins = 0
    t0_all = time.time()

    for i, task in enumerate(task_list):
        total += 1
        t0 = time.time()

        predictions = solve_task_hybrid(task, model, device,
                                       dsl_time=dsl_time, ttt_steps=ttt_steps)

        # Score: all test outputs must match
        task_correct = True
        for preds, tp in zip(predictions, task.test):
            if tp.output is None:
                task_correct = False  # Can't score without ground truth
                continue
            if not any(np.array_equal(p, tp.output) for p in preds):
                task_correct = False

        if task_correct:
            correct += 1

        elapsed = time.time() - t0
        if (i + 1) % 10 == 0:
            print(f'  [{i+1}/{len(task_list)}] Correct: {correct}/{total} = {correct/total:.1%}  '
                  f'({elapsed:.1f}s this task, {time.time()-t0_all:.0f}s total)')

    rate = correct / max(total, 1)
    print(f'\n=== FINAL: {correct}/{total} = {rate:.1%} ===')
    print(f'Total time: {time.time()-t0_all:.0f}s')
    return correct, total, rate

# Run evaluation
print('=== Evaluating on eval set ===')
correct, total, rate = evaluate_hybrid(
    eval_tasks, neural_model, DEVICE,
    max_tasks=50,  # Start with 50 for speed; set to None for full eval
    dsl_time=20.0,
    ttt_steps=50,
)

In [ ]:
# ============================================================
# Cell 13: Generate Submission
# ============================================================

def generate_submission(tasks: Dict[str, ARCTask], model, device,
                        output_path='submission.json',
                        dsl_time=30.0, ttt_steps=75):
    """Generate Kaggle submission file."""
    submission = {}
    task_list = list(tasks.values())
    t0 = time.time()

    for i, task in enumerate(task_list):
        predictions = solve_task_hybrid(task, model, device,
                                       dsl_time=dsl_time, ttt_steps=ttt_steps)
        task_attempts = []
        for preds in predictions:
            task_attempts.append({
                'attempt_1': preds[0].astype(int).tolist(),
                'attempt_2': preds[1].astype(int).tolist() if len(preds) > 1 else preds[0].astype(int).tolist(),
            })
        submission[task.task_id] = task_attempts

        if (i + 1) % 25 == 0:
            elapsed = time.time() - t0
            print(f'  [{i+1}/{len(task_list)}] {elapsed:.0f}s elapsed')

    with open(output_path, 'w') as f:
        json.dump(submission, f)

    print(f'Submission saved to {output_path} ({len(submission)} tasks, {time.time()-t0:.0f}s)')
    return submission

# For Kaggle: generate submission on test tasks
if test_tasks:
    print('=== Generating competition submission ===')
    sub = generate_submission(test_tasks, neural_model, DEVICE)
else:
    # For local dev: generate on eval tasks to measure score
    print('No test tasks found (not on Kaggle). Skipping submission generation.')
    print('Run full eval above to measure performance.')

---
## Next Steps to Push Toward 85%

This notebook implements the foundation. To push toward the Grand Prize:

### DSL Improvements (biggest ROI)
1. **More primitives**: flood fill, line drawing, pattern completion, symmetry detection, grid splitting/recombining
2. **Pattern-matching heuristics**: detect repeated subgrids, detect color mappings, detect border/interior relationships
3. **Conditional programs**: if-then-else based on grid properties (e.g., "if grid has symmetry, complete it; else rotate")
4. **Object-level DSL**: operate on extracted objects (move, copy, recolor, sort by size/color/position)

### Neural Improvements
5. **Synthetic data pre-training**: Generate millions of synthetic ARC-like tasks for pre-training (this is what NVARC did)
6. **Better architecture**: Transformer-based encoder instead of CNN, autoregressive decoder
7. **Augmentation ensembling**: Solve with multiple augmented views, vote on the answer
8. **Output size prediction**: Separate head to predict output grid dimensions

### Hybrid Improvements
9. **Neural-guided search**: Use neural model features to rank which DSL primitives to try first
10. **LLM program synthesis**: Use a local LLM to generate Python programs (if allowed within constraints)
11. **Multi-attempt diversity**: Make the 2 candidates genuinely different (DSL + neural, or two different TTT runs)